In [4]:
import pandas as pd
import numpy as np
import json
order_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv",
]

df_orders = pd.concat(
    [pd.read_csv(f, parse_dates=["order_datetime"]) for f in order_files],
    ignore_index=True
)


with open("data/events_records.json", encoding="utf-8") as f:
    raw = json.load(f)

n = len(raw[0])
df_records = pd.DataFrame({
    "event_id":   list(range(1, n + 1)),
    "user_id":    [raw[1][str(i)] for i in range(n)],
    "event_time": [raw[2][str(i)] for i in range(n)],
    "event_type": [raw[3][str(i)] for i in range(n)],
    "page":       [raw[4][str(i)] for i in range(n)],
})


df_lines = pd.read_json("data/events_lines.json", lines=True)


df_events = pd.concat([df_records, df_lines], ignore_index=True)
df_events["event_time"] = pd.to_datetime(df_events["event_time"])

events_users = pd.DataFrame({
    "user_id": df_events["user_id"].unique()
})
events_users["has_events"] = 1

print("Пользователей с событиями:", len(events_users))
print(events_users.head())

Пользователей с событиями: 100
   user_id  has_events
0     1085           1
1     1006           1
2     1024           1
3     1062           1
4     1012           1


In [5]:
orders_users = pd.DataFrame({
    "user_id": df_orders["user_id"].unique()
})
orders_users["has_orders"] = 1

print("Пользователей с заказами:", len(orders_users))
print(orders_users.head())

Пользователей с заказами: 100
   user_id  has_orders
0     1044           1
1     1079           1
2     1041           1
3     1083           1
4     1068           1


In [6]:
user_flags = orders_users.merge(
    events_users,
    on="user_id",
    how="outer"
)


user_flags["has_events"] = user_flags["has_events"].fillna(0).astype(int)
user_flags["has_orders"] = user_flags["has_orders"].fillna(0).astype(int)

print("Всего уникальных пользователей:", len(user_flags))
print(user_flags.head())

crosstab = pd.crosstab(
    user_flags["has_events"],
    user_flags["has_orders"],
    margins=True,
    margins_name="Итого"
)

print("\nПерекрёстная таблица (has_events × has_orders):")
print(crosstab)

Всего уникальных пользователей: 100
   user_id  has_orders  has_events
0     1001           1           1
1     1002           1           1
2     1003           1           1
3     1004           1           1
4     1005           1           1

Перекрёстная таблица (has_events × has_orders):
has_orders    1  Итого
has_events            
1           100    100
Итого       100    100
